# 00_runtime_mode_and_user_config

In [ ]:
from __future__ import annotations
import json
import os
import platform
import subprocess
import sys
from pathlib import Path
RUN_MODE = 'smoke'
SAMPLES_PER_ROLE = 1
RUNTIME_PROFILE = 'tiny'
DRIVE_ROOT = Path('/content/drive/MyDrive/tstw_stage2')
REPO_ROOT = Path('/content/TSTW-VW')
RUNS_ROOT = DRIVE_ROOT / 'runs'
PACKED_ROOT = DRIVE_ROOT / 'packed'
SMOKE_RUN_ROOT = RUNS_ROOT / 'stage2_smoke'
FORMAL_RUN_ROOT = RUNS_ROOT / 'stage2_formal'
print({'run_mode': RUN_MODE, 'runtime_profile': RUNTIME_PROFILE, 'drive_root': str(DRIVE_ROOT)})

# 01_mount_google_drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 02_clone_or_update_repository

In [ ]:
REPO_URL = os.environ.get('TSTW_VW_REPO_URL', 'https://github.com/your-org/TSTW-VW.git')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
print(subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

# 03_install_dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest'], check=True)

# 04_verify_repository_contract

In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], cwd=REPO_ROOT, check=True)

# 05_prepare_drive_directories

In [ ]:
for directory in (DRIVE_ROOT, RUNS_ROOT, PACKED_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print(str(RUNS_ROOT))

# 06_write_or_select_stage2_configs

In [ ]:
runtime_config_path = DRIVE_ROOT / 'colab_stage2_runtime_config.json'
git_commit = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
runtime_config_payload = {'git_commit': git_commit, 'runtime_origin': 'colab', 'drive_root': str(DRIVE_ROOT), 'config_file_name': 'colab_stage2_runtime_config.json'}
runtime_config_path.write_text(json.dumps(runtime_config_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(str(runtime_config_path))

# 07_prepare_or_validate_video_dataset

In [ ]:
dataset_manifest = REPO_ROOT / 'configs' / 'data' / 'real_video_probe_manifest.json'
if not dataset_manifest.exists():
    raise FileNotFoundError(dataset_manifest)
print(str(dataset_manifest))

# 08_check_gpu_and_runtime

In [ ]:
gpu_name = 'not_available'
try:
    gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
except Exception:
    pass
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'gpu': gpu_name})

# 09_load_vae_model

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from main.core.registry import load_json_config
from main.vae.vae_registry import resolve_vae_backend
backend_config = load_json_config(REPO_ROOT / 'configs' / 'backend' / 'real_video_vae_latent.json')
vae_backend = resolve_vae_backend(backend_config)
print(vae_backend.backend_metadata())

# 10_run_unit_tests_smoke

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-m', 'smoke', 'tests/test_stage2_records_schema.py', 'tests/test_stage2_table_rebuild.py', 'tests/test_stage2_drive_packager.py'], cwd=REPO_ROOT, check=True)

# 11_run_stage2_smoke

In [ ]:
from main.protocol.stage2_runner import Stage2Runner
runner = Stage2Runner(REPO_ROOT)
smoke_result = runner.run(output_root=SMOKE_RUN_ROOT, run_mode='smoke', samples_per_role=SAMPLES_PER_ROLE, runtime_profile_override=RUNTIME_PROFILE, runtime_config_path=runtime_config_path)
print({'run_id': smoke_result.run_id, 'output_root': str(smoke_result.output_root)})

# 12_check_stage2_smoke_outputs

In [ ]:
from main.colab.notebook_result_checker import check_stage2_outputs
smoke_checks = check_stage2_outputs(SMOKE_RUN_ROOT, run_mode='smoke')
if not smoke_checks['status']:
    raise RuntimeError(smoke_checks)
print(smoke_checks)

# 13_run_stage2_formal

In [ ]:
formal_result = runner.run(output_root=FORMAL_RUN_ROOT, run_mode='formal', samples_per_role=max(2, SAMPLES_PER_ROLE), runtime_profile_override='formal', runtime_config_path=runtime_config_path)
print({'run_id': formal_result.run_id, 'output_root': str(formal_result.output_root)})

# 14_rebuild_tables_and_reports

In [ ]:
from main.analysis.stage2_artifacts import Stage2ArtifactBuilder
rebuilt_paths = Stage2ArtifactBuilder().rebuild_artifacts(FORMAL_RUN_ROOT)
print({key: str(value) for key, value in rebuilt_paths.items()})

# 15_validate_stage2_results

In [ ]:
formal_checks = check_stage2_outputs(FORMAL_RUN_ROOT, run_mode='formal', require_formal_pass_criteria=False)
print(formal_checks)

# 16_pack_results_to_drive

In [ ]:
from main.colab.drive_packager import pack_stage2_run
packaged_paths = pack_stage2_run(run_root=FORMAL_RUN_ROOT, drive_output_dir=PACKED_ROOT, include_records=True, include_thresholds=True, include_tables=True, include_figures=True, include_reports=True, include_failure_gallery=True, include_manifest=True)
print({key: str(value) for key, value in packaged_paths.items()})

# 17_print_final_summary

In [ ]:
summary = {'smoke_run_root': str(SMOKE_RUN_ROOT), 'formal_run_root': str(FORMAL_RUN_ROOT), 'packed_root': str(PACKED_ROOT), 'runtime_config': str(runtime_config_path)}
print(json.dumps(summary, ensure_ascii=False, indent=2))